# 04 - Leakage: when 99 % is a warning sign

**Data leakage** means the model sees information it would not have when the prediction is actually needed, or information that is a rewording of the answer. The result looks wonderful in the notebook and is useless in real life.

You will see four kinds of leakage on this data:

1. **Outcome columns**: stats that only exist because the game already ended.
2. **A target computed from the features**: predicting a formula is not machine learning.
3. **Row split instead of match split**: players of the same match sit in train *and* test.
4. **Near-duplicate rows**: snapshots of the same player a few seconds apart.

In [ ]:
import os, sys, json, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # make the `lolml` helpers importable (notebooks run from notebooks/)
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from lolml import data, features as F, paths
paths.ensure_dirs()
SMOKE = bool(os.environ.get("LOLML_SMOKE"))   # LOLML_SMOKE=1 uses the small committed sample and short training budgets
SEED = 42
def save_fig(name):
    plt.tight_layout(); plt.savefig(paths.OUT / name, dpi=110); plt.show()

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, r2_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
ITER = 60 if SMOKE else 150

## 1. The leakage ladder

Same model, same split by match, four feature sets. Each rung removes something a real predictor would not have.

In [ ]:
perf = data.load("performance")
perf["win"] = perf["win"].astype(int)
tr, te = F.group_split(perf, 0.2, SEED)
train, test = perf.iloc[tr], perf.iloc[te]

OBJECTIVES = ["damageDealtToBuildings", "damageDealtToObjectives", "damageDealtToTurrets", "turretKills", "turretTakedowns",
              "dragonKills", "baronKills", "objectivesStolen", "objectivesStolenAssists", "firstTowerKill", "firstTowerAssist"]
everything = F.post_game_numeric(perf, drop_outcome=False)
no_outcome = F.post_game_numeric(perf)
no_objectives = [c for c in no_outcome if c not in OBJECTIVES]
ladder = {
    "1. everything (incl. nexus, inhibitors, surrender)": everything,
    "2. drop the outcome columns": no_outcome,
    "3. also drop team objectives": no_objectives,
    "4. only kills, deaths, assists": ["kills", "deaths", "assists"],
    "(the outcome columns on their own)": F.OUTCOME_LEAKS,
}
rows = []
for name, c in ladder.items():
    m = HistGradientBoostingClassifier(max_iter=ITER, random_state=SEED).fit(train[c], train["win"])
    rows.append((name, len(c), accuracy_score(test["win"], m.predict(test[c]))))
ladder_df = pd.DataFrame(rows, columns=["feature set", "features", "accuracy"]).set_index("feature set")
print(ladder_df.round(4).to_string())
ladder_df.to_json(paths.OUT / "leakage_ladder.json")

In [ ]:
ax = ladder_df["accuracy"][::-1].plot.barh(figsize=(8, 3.5), color="#2d5967", xlim=(0.5, 1.0), title="Accuracy on unseen matches")
for i, v in enumerate(ladder_df["accuracy"][::-1]): ax.text(v + 0.005, i, f"{v:.1%}", va="center")
save_fig("leakage_ladder.png")

Rung 1 looks like a miracle: five columns about the nexus and inhibitors (`nexusKills`, `inhibitorsLost`, ...) are almost a statement of who won. Rung 2 is already a different story, but it is still a **post-game explainer**, because objective stats such as `turretKills` need a won lane to exist. Only rung 3 or 4 describes what one player did, and even that cannot be known at minute 5.

## 2. A target that is a formula

The original workshop trained a regressor on `calculated_player_performance` and celebrated an R² of 99.99 %. That column is a hand-written formula of per-minute stats (see `src/process_performance.py`):

`100 * (0.336 - 1.437*deaths/min + 0.000117*gold/min + 0.443*(kills+assists)/min + 0.264*level/min + 0.000013*damage/min)`

Ordinary linear regression on those five ingredients finds it exactly:

In [ ]:
d = perf.dropna(subset=["duration"]).copy()
mins = d["duration"]
ingredients = pd.DataFrame({"deaths_pm": d.deaths / mins, "gold_pm": d.goldEarned / mins, "ka_pm": (d.kills + d.assists) / mins,
                            "level_pm": d.champLevel / mins, "damage_pm": d.totalDamageDealt / mins})
lin = LinearRegression().fit(ingredients, d["calculated_player_performance"])
print("R^2 of plain linear regression:", round(r2_score(d["calculated_player_performance"], lin.predict(ingredients)), 4))
print({k: round(float(v), 6) for k, v in zip(ingredients.columns, lin.coef_ / 100)}, "intercept", round(float(lin.intercept_) / 100, 3))

The fitted weights are the constants of the formula (0.336, -1.437, 0.000117, 0.443, 0.264, 0.000013, up to small differences caused by how game length is measured). AutoGluon "predicting performance with 99.99 % accuracy" only rediscovers a formula somebody typed in. Nothing was predicted.

## 3. Row split versus match split

The by-minute data has about 286 snapshots per match: 10 players, every few seconds. Shuffle the rows and the test set contains snapshots from matches the model trained on.

In [ ]:
bym = data.load("by_minute")
bc = F.BYMIN_LIVE_SAFE
out = []
for split_name, (a, b) in (("row split (shuffled rows)", F.row_split(bym, 0.2, SEED)), ("match split", F.group_split(bym, 0.2, SEED))):
    a_, b_ = bym.iloc[a], bym.iloc[b]
    hgb = HistGradientBoostingClassifier(max_iter=ITER, random_state=SEED).fit(a_[bc], a_["winner"])
    knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=1, n_jobs=-1)).fit(a_[bc], a_["winner"])
    out.append((split_name, accuracy_score(b_["winner"], hgb.predict(b_[bc])), accuracy_score(b_["winner"], knn.predict(b_[bc]))))
split_df = pd.DataFrame(out, columns=["split", "gradient boosting", "1-nearest-neighbour"]).set_index("split")
print(split_df.round(4).to_string())
split_df.to_json(paths.OUT / "split_comparison.json")

Gradient boosting is only a little optimistic with a row split. The 1-nearest-neighbour model, which does little more than look up the closest row it has seen, gains several points on shuffled rows: its nearest neighbour is the same player a few seconds earlier, and it carries the answer. Always split by the thing that defines "a new game": the match.

## 4. A checklist before you believe a number

* Would I have this column **at the moment I need the prediction**?
* Is any feature computed from the target, or from a column that is?
* Do rows from the same match, player or session appear on both sides of the split?
* Is the score too good? Compare with a trivial baseline (always predict "win" = 50 %).
* Does the score change a lot when I drop the single most important feature?